In [2]:
import pandas as pd
import numpy as np

train = pd.read_parquet('../data/raw/train_parsed.parquet')

daily = (
    train.groupby(['Date', 'ItemCode'])['Quantity']
    .sum()
    .clip(lower=0)
    .reset_index()
    .rename(columns={'Quantity': 'Qty'})
)
daily = daily[daily['Qty'] > 0]

all_dates = pd.date_range('2020-11-17', '2025-09-05', freq='D')
all_skus  = sorted(train['ItemCode'].unique())

grid = (
    daily.pivot_table(index='Date', columns='ItemCode',
                      values='Qty', fill_value=0)
    .reindex(index=all_dates, columns=all_skus, fill_value=0)
)

grid.to_parquet('../data/processed/daily_sales.parquet')
print(f"Grid saved: {grid.shape}")

Grid saved: (1754, 15972)
